# WP-02 PoC — TodoTango Metadata Scraper

**Goal:** Enrich all local MP3s in `data/raw/` with authoritative metadata from [todotango.com](https://www.todotango.com):
genre, composer, lyricist, recording date, city, label, catalog number, and vocalist info.

**Output:** `data/processed/todotango_enriched.csv` — one row per MP3, ready to merge into `catalog.csv`.

**Runtime note:**
- First run: ~70 min for 942 tracks (2 HTTP requests each at 1.5 s rate limit)
- Re-runs using disk cache: ~2 min
- Resume-safe: restart the batch cell at any time — already-processed files are skipped

---

## 0. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import re
import time
import hashlib
from pathlib import Path
from difflib import SequenceMatcher

import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
from mutagen.id3 import ID3

from atdj.config import SAMPLES_DIR, RAW_DIR, PROCESSED_DIR

# --- Cache & rate limit ---
CACHE_DIR  = PROCESSED_DIR / "todotango_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RATE_LIMIT = 1.5   # seconds between live HTTP requests

# --- URLs ---
BASE_URL   = "https://www.todotango.com"
SEARCH_URL = "https://www.todotango.com/buscar/?kwd={query}"

# --- Session headers (polite bot identification) ---
SESSION_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-AR,es;q=0.9,en;q=0.8",
}

print(f"RAW_DIR:   {RAW_DIR}")
print(f"CACHE_DIR: {CACHE_DIR}")
print(f"Output:    {PROCESSED_DIR / 'todotango_enriched.csv'}")

---
## 1. Load Local Files

Extract title and orchestra from ID3 tags for every MP3 in `data/raw/`.
Falls back to filename stem for title when tags are missing.

In [ ]:
def read_local_metadata(mp3_path: Path) -> dict:
    """Read title and orchestra from ID3 tags; fallback to filename stem."""
    result = {
        "file_path":       str(mp3_path),
        "filename":        mp3_path.name,
        "local_title":     mp3_path.stem,
        "local_orchestra": "",
    }
    try:
        tags = ID3(mp3_path)
        title = str(tags.get("TIT2", "")).strip()
        orch  = str(tags.get("TPE1", "")).strip()
        if title:
            result["local_title"] = title
        if orch:
            result["local_orchestra"] = orch
    except Exception:
        pass
    return result


records  = [read_local_metadata(p) for p in sorted(RAW_DIR.glob("*.mp3"))
            if not p.name.startswith("._")]
df_local = pd.DataFrame(records)

print(f"Total local MP3s: {len(df_local)}")
print(f"Missing title tag:     {(df_local['local_title'] == df_local['filename'].str.replace('.mp3','')).sum()} (fell back to filename)")
print(f"Missing orchestra tag: {(df_local['local_orchestra'] == '').sum()}")
df_local.head(5)

---
## 2. Core Scraping Functions

In [ ]:
# --- 2a. Orchestra normalisation (run on BOTH sides before fuzzy match) ---

def normalize_orchestra(name: str) -> str:
    """
    Strip common prefixes/suffixes so fuzzy matching works across naming conventions.
    Examples:
      'Carlos Di Sarli (Instrumental)'  -> 'Carlos Di Sarli'
      'Orquesta Típica Victor'           -> 'Típica Victor'
      'Los Tubatango'                    -> 'Tubatango'
    """
    name = re.sub(r'\s*\(.*?\)', '', name)             # remove (Instrumental), (con canto) etc.
    name = re.sub(r'^Orquesta\s+', '', name, flags=re.IGNORECASE)
    name = re.sub(r'^Los\s+',      '', name, flags=re.IGNORECASE)
    name = re.sub(r'^Las\s+',      '', name, flags=re.IGNORECASE)
    return name.strip()

In [ ]:
# --- 2b. Caching HTTP helper ---

def _get_cached_or_fetch(url: str, session: requests.Session) -> str:
    """
    Return HTML for url.
    Writes raw HTML to CACHE_DIR/{md5(url)}.html on first fetch;
    returns the cached string on subsequent calls (no sleep, no network).
    """
    url_hash   = hashlib.md5(url.encode()).hexdigest()
    cache_file = CACHE_DIR / f"{url_hash}.html"
    if cache_file.exists():
        return cache_file.read_text(encoding="utf-8", errors="replace")
    time.sleep(RATE_LIMIT)
    resp = session.get(url, timeout=15)
    resp.raise_for_status()
    html = resp.text
    cache_file.write_text(html, encoding="utf-8")
    return html

In [ ]:
# --- 2c. Search ---

def search_todotango(query: str, session: requests.Session) -> list[dict]:
    """
    Search todotango.com for a track title.
    Returns list of {'title': str, 'url': str}.
    Returns [] on no results or HTTP error.

    Selector chain:
      container : #main_buscar1_pnl_Temas
      each link : .row .col-xs-12 h3 a
      href      : /musica/tema/{ID}/{SLUG}/
    """
    url  = SEARCH_URL.format(query=requests.utils.quote(query))
    try:
        html = _get_cached_or_fetch(url, session)
    except Exception:
        return []
    soup      = BeautifulSoup(html, "html.parser")
    container = soup.select_one("#main_buscar1_pnl_Temas")
    if container is None:
        return []
    results = []
    for a in container.select(".row .col-xs-12 h3 a"):
        href = a.get("href", "")
        if "/musica/tema/" in href:
            full_url = href if href.startswith("http") else BASE_URL + href
            results.append({"title": a.get_text(strip=True), "url": full_url})
    return results

In [ ]:
# --- 2d. Parse recording details string ---

# Known multi-word cities (checked before splitting on spaces)
_KNOWN_CITIES = ["Buenos Aires", "Montevideo", "New York", "Paris", "London"]

def parse_recording_details(details_str: str) -> dict:
    """
    Parse the lbl_DetallesGrabacion span text.
    Format: 'DD-MM-YYYY CITY LABEL CATALOG1 [CATALOG2]'
    Example: '19-12-1936 Buenos Aires Odeon 3503 8869'

    Returns keys: recording_date, recording_year, recording_city,
                  recording_label, recording_catalog
    """
    out = {
        "recording_date":    None,
        "recording_year":    None,
        "recording_city":    None,
        "recording_label":   None,
        "recording_catalog": None,
    }
    if not details_str:
        return out

    s = details_str.strip()

    # Extract date (DD-MM-YYYY) or bare year (YYYY)
    date_m = re.match(r'(\d{1,2}-\d{1,2}-(\d{4}))', s)
    if date_m:
        out["recording_date"] = date_m.group(1)
        out["recording_year"] = date_m.group(2)
        s = s[date_m.end():].strip()
    else:
        year_m = re.search(r'\b(19\d{2}|20\d{2})\b', s)
        if year_m:
            out["recording_year"] = year_m.group(1)

    # Extract city
    for city in _KNOWN_CITIES:
        if city.lower() in s.lower():
            out["recording_city"] = city
            s = re.sub(re.escape(city), "", s, flags=re.IGNORECASE).strip()
            break

    # Remaining: first token = label, rest = catalog
    tokens = s.split()
    if tokens:
        out["recording_label"]   = tokens[0]
        out["recording_catalog"] = " ".join(tokens[1:]) if len(tokens) > 1 else None

    return out

In [ ]:
# --- 2e. Fetch & parse track detail page ---

def fetch_track_detail(url: str, session: requests.Session) -> dict:
    """
    Fetch a todotango track page and extract all metadata.

    Top-level fields:
      Title       : #main_Tema1_lbl_Titulo
      Genre       : #main_Tema1_lbl_Ritmo           e.g. 'Tango', 'Vals', 'Milonga'
      Comp. year  : #main_Tema1_lbl_Ano
      Composer    : [id*='AutoresMusica_hl_Creador'] (first match)
      Lyricist    : [id*='AutoresLetra_hl_Creador']  (first match)

    Each recording block (div.cajita_gris3):
      Genre+Dur   : [id*='lbl_RitmoDuracion_']  e.g. "Tango 02'14\""
      Vocalist    : [id*='lbl_Canta_']           e.g. 'Canta Raúl Berón' or 'Canta Instrumental'
      Formation   : [id*='lbl_Formacion_']        e.g. 'Orquesta', 'Trío'
      Orchestra   : [id*='lbl_Interprete_']       e.g. 'Carlos Di Sarli'
      Details     : [id*='lbl_DetallesGrabacion_'] e.g. '19-12-1936 Buenos Aires Odeon 3503 8869'
    """
    html = _get_cached_or_fetch(url, session)
    soup = BeautifulSoup(html, "html.parser")

    def text(selector: str) -> str | None:
        el = soup.select_one(selector)
        return el.get_text(strip=True) if el else None

    tt_id_m = re.search(r'/musica/tema/(\d+)/', url)
    tt_id   = tt_id_m.group(1) if tt_id_m else None

    result = {
        "tt_url":           url,
        "tt_id":            tt_id,
        "tt_title":         text("#main_Tema1_lbl_Titulo"),
        "genre":            text("#main_Tema1_lbl_Ritmo"),
        "composition_year": text("#main_Tema1_lbl_Ano"),
        "composer":         text('[id*="AutoresMusica_hl_Creador"]'),
        "lyricist":         text('[id*="AutoresLetra_hl_Creador"]'),
        "versions":         [],
    }

    for block in soup.select("div.cajita_gris3"):
        def btext(frag: str) -> str | None:
            el = block.select_one(f'[id*="{frag}"]')
            return el.get_text(strip=True) if el else None

        raw_ritmo     = btext("lbl_RitmoDuracion_")
        raw_canta     = btext("lbl_Canta_")
        raw_formacion = btext("lbl_Formacion_")
        raw_interprete= btext("lbl_Interprete_")
        raw_detalles  = btext("lbl_DetallesGrabacion_")

        # Split "Tango 02'14\"" into genre + duration
        ver_genre = ver_duration = None
        if raw_ritmo:
            parts = raw_ritmo.split(None, 1)
            ver_genre    = parts[0]
            ver_duration = parts[1] if len(parts) > 1 else None

        # Strip leading 'Canta ' prefix from vocalist
        vocalist = re.sub(r'^Canta\s+', '', raw_canta or "", flags=re.IGNORECASE).strip() or None

        recording = parse_recording_details(raw_detalles or "")

        result["versions"].append({
            "version_genre":      ver_genre,
            "version_duration":   ver_duration,
            "version_vocal":      vocalist,
            "version_formation":  raw_formacion,
            "version_orchestra":  raw_interprete,
            **recording,
        })

    return result

In [ ]:
# --- 2f. Matching helpers ---

def _similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a.lower().strip(), b.lower().strip()).ratio()


def pick_best_search_result(results: list[dict], local_title: str) -> dict | None:
    """Pick the search result with the highest title similarity to local_title."""
    if not results:
        return None
    if len(results) == 1:
        return results[0]
    return max(results, key=lambda r: _similarity(local_title, r["title"]))


def match_best_version(
    versions: list[dict], local_orchestra: str
) -> tuple[dict | None, float]:
    """
    Find the version whose orchestra best matches local_orchestra.
    Returns (best_version, confidence 0.0–1.0).

    Logic:
      - 0 versions  → (None, 0.0)
      - 1 version   → (versions[0], 1.0)  — take it unconditionally
      - multiple    → normalize both sides, pick highest SequenceMatcher ratio
    """
    if not versions:
        return None, 0.0
    if len(versions) == 1:
        return versions[0], 1.0

    local_norm = normalize_orchestra(local_orchestra).lower()
    best_ver, best_ratio = None, 0.0
    for v in versions:
        orch_norm = normalize_orchestra(v.get("version_orchestra") or "").lower()
        ratio = _similarity(local_norm, orch_norm)
        if ratio > best_ratio:
            best_ratio, best_ver = ratio, v
    return best_ver, best_ratio

In [ ]:
# --- 2g. Main enrichment entry point ---

_EMPTY_OUTPUT = {
    "tt_url": None, "tt_id": None, "tt_title": None,
    "genre": None, "composition_year": None,
    "composer": None, "lyricist": None,
    "version_vocal": None, "version_formation": None,
    "version_orchestra": None, "recording_date": None,
    "recording_year": None, "recording_city": None,
    "recording_label": None, "recording_catalog": None,
    "version_duration": None,
    "match_confidence": 0.0,
    "match_status": "not_found",
}


def enrich_track(mp3_path: Path, session: requests.Session) -> dict:
    """
    Full pipeline for one MP3:
      1. Read local metadata (title, orchestra from ID3)
      2. Search todotango by title
      3. Pick best search result by title similarity
      4. Fetch track detail page
      5. Match best version by orchestra fuzzy match
      6. Return flat dict with all fields + match_status

    match_status:
      'found'     — confidence >= 0.4
      'ambiguous' — hit found but version confidence < 0.4
      'not_found' — no search results
      'error'     — HTTP or parse exception
    """
    local  = read_local_metadata(mp3_path)
    output = {**local, **_EMPTY_OUTPUT}

    try:
        results     = search_todotango(local["local_title"], session)
        best_result = pick_best_search_result(results, local["local_title"])

        if best_result is None:
            output["match_status"] = "not_found"
            return output

        detail = fetch_track_detail(best_result["url"], session)
        output.update({
            "tt_url":           detail["tt_url"],
            "tt_id":            detail["tt_id"],
            "tt_title":         detail["tt_title"],
            "genre":            detail["genre"],
            "composition_year": detail["composition_year"],
            "composer":         detail["composer"],
            "lyricist":         detail["lyricist"],
        })

        best_ver, confidence = match_best_version(
            detail["versions"], local["local_orchestra"]
        )
        if best_ver:
            output.update({
                "version_vocal":      best_ver["version_vocal"],
                "version_formation":  best_ver["version_formation"],
                "version_orchestra":  best_ver["version_orchestra"],
                "recording_date":     best_ver["recording_date"],
                "recording_year":     best_ver["recording_year"],
                "recording_city":     best_ver["recording_city"],
                "recording_label":    best_ver["recording_label"],
                "recording_catalog":  best_ver["recording_catalog"],
                "version_duration":   best_ver["version_duration"],
                "match_confidence":   round(confidence, 3),
                "match_status":       "found" if confidence >= 0.4 else "ambiguous",
            })
        else:
            output["match_status"] = "ambiguous"

    except Exception as exc:
        output["match_status"] = "error"
        output["tt_title"]     = str(exc)   # store error message for review

    return output


print("All scraping functions defined.")

---
## 3. Test on 3 Sample Files

Run the full pipeline on the 3 tango samples in `data/samples/` before touching the full library.
Expected: `match_status=found`, genre populated, recording date matches known discography.

In [ ]:
from IPython.display import Audio, display

session = requests.Session()
session.headers.update(SESSION_HEADERS)

test_files = sorted(
    [p for p in SAMPLES_DIR.glob("*.mp3")
     if not p.name.startswith("._") and "cortina" not in p.name.lower()]
)[:3]

test_results = []
for f in test_files:
    print(f"\n{'='*60}")
    print(f"File   : {f.name}")
    display(Audio(str(f)))

    r = enrich_track(f, session)
    test_results.append(r)

    print(f"Status : {r['match_status']}  (confidence={r['match_confidence']})")
    print(f"Title  : {r['local_title']}  →  {r['tt_title']}")
    print(f"Genre  : {r['genre']}")
    print(f"Composer : {r['composer']}  |  Lyricist: {r['lyricist']}")
    print(f"Orchestra: {r['version_orchestra']}  |  Vocalist: {r['version_vocal']}")
    print(f"Recorded : {r['recording_date']}  {r['recording_city']}  {r['recording_label']}  {r['recording_catalog']}")

In [ ]:
# All fields in a transposed table (easy to read when columns >> rows)
df_test = pd.DataFrame(test_results).set_index("filename")
df_test.drop(columns=["file_path"]).T

---
## 4. Batch Processing — All Files in data/raw/

**Resume-safe:** already-processed filenames are skipped on restart.

**Progress:** prints every 10 tracks; flushes to CSV every 50 records.

> First run: ~70 min (942 tracks × 2 requests × 1.5 s).  
> Re-runs from cache: ~2 min.

In [ ]:
OUTPUT_CSV  = PROCESSED_DIR / "todotango_enriched.csv"
WRITE_EVERY = 50

all_mp3s = [p for p in sorted(RAW_DIR.glob("*.mp3")) if not p.name.startswith("._")]
print(f"Total files: {len(all_mp3s)}")

# Resume: skip files already in the output CSV
processed_filenames: set[str] = set()
if OUTPUT_CSV.exists():
    processed_filenames = set(pd.read_csv(OUTPUT_CSV)["filename"].tolist())
    print(f"Resuming — {len(processed_filenames)} already processed")

remaining = [p for p in all_mp3s if p.name not in processed_filenames]
print(f"Remaining : {len(remaining)}")

In [ ]:
buffer: list[dict] = []

for i, mp3_path in enumerate(remaining):
    result = enrich_track(mp3_path, session)
    buffer.append(result)

    if (i + 1) % 10 == 0 or (i + 1) == len(remaining):
        status_icon = {"found": "✓", "ambiguous": "?", "not_found": "✗", "error": "!"}
        icon = status_icon.get(result["match_status"], "?")
        print(f"  [{i+1:4d}/{len(remaining)}] {icon} {mp3_path.name[:50]}")

    if len(buffer) >= WRITE_EVERY:
        chunk = pd.DataFrame(buffer)
        chunk.to_csv(OUTPUT_CSV, mode="a", header=not OUTPUT_CSV.exists(), index=False)
        buffer.clear()

# Final flush
if buffer:
    chunk = pd.DataFrame(buffer)
    chunk.to_csv(OUTPUT_CSV, mode="a", header=not OUTPUT_CSV.exists(), index=False)

print(f"\nBatch complete. Output: {OUTPUT_CSV}")

---
## 5. Review Results

In [ ]:
df_enriched = pd.read_csv(OUTPUT_CSV)
print(f"Total rows: {len(df_enriched)}")
print()

print("=== Match status ===")
print(df_enriched["match_status"].value_counts())
print()

print("=== Genre breakdown (found tracks) ===")
print(df_enriched[df_enriched["match_status"] == "found"]["genre"].value_counts())

In [ ]:
# Confidence score distribution
found = df_enriched[df_enriched["match_status"] == "found"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

found["match_confidence"].hist(bins=20, ax=axes[0], color="steelblue")
axes[0].set_title("Version match confidence (found tracks)")
axes[0].set_xlabel("Confidence (difflib ratio)")
axes[0].set_ylabel("Count")

found["recording_year"].dropna().astype(int).hist(bins=20, ax=axes[1], color="coral")
axes[1].set_title("Recording year distribution")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Tracks that need manual review
not_found = df_enriched[df_enriched["match_status"] == "not_found"]
print(f"=== Not found ({len(not_found)}) ===")
if len(not_found):
    print(not_found[["filename", "local_title", "local_orchestra"]].to_string(index=False))

print()
ambiguous = df_enriched[df_enriched["match_status"] == "ambiguous"]
print(f"=== Ambiguous — low version confidence ({len(ambiguous)}) ===")
if len(ambiguous):
    print(ambiguous[["filename", "local_title", "local_orchestra",
                      "version_orchestra", "match_confidence"]].to_string(index=False))

In [ ]:
# Spot-check 5 random found tracks
df_enriched[df_enriched["match_status"] == "found"].sample(5)[
    ["local_title", "local_orchestra", "tt_title",
     "genre", "version_orchestra", "recording_date",
     "recording_label", "recording_catalog", "match_confidence"]
]

---
## 6. Export

The enriched CSV is already written incrementally during the batch step.
This cell saves a clean final copy and prints the column manifest.

### Merge mapping to `catalog.csv` (deferred to `04_catalog_merge.ipynb`)

| Scraper column | catalog.csv column | Notes |
|---|---|---|
| `genre` (lowercased) | `style` | Tango→tango, Vals→vals, Milonga→milonga |
| `composition_year` | `year` | Only where `year` is currently null |
| `version_vocal` | `singer` | 'Instrumental' → leave singer empty |
| `recording_year` | — | New column |
| `recording_city` | — | New column |
| `recording_label` | — | New column |
| `recording_catalog` | — | New column |
| `composer` | — | New column |
| `lyricist` | — | New column |
| `tt_url` | — | Provenance link |

In [ ]:
FINAL_CSV = PROCESSED_DIR / "todotango_enriched.csv"
df_enriched.to_csv(FINAL_CSV, index=False)

print(f"Saved: {FINAL_CSV}")
print(f"Rows   : {len(df_enriched)}")
print(f"Columns: {len(df_enriched.columns)}")
print()
print("Column manifest:")
for col in df_enriched.columns:
    non_null = df_enriched[col].notna().sum()
    print(f"  {col:30s}  {non_null:4d} / {len(df_enriched)} non-null")